# Chapter 11: Candidate Generation as a Pipeline

Union, provenance and the candidate recall ceiling.

In [ ]:
import sys
from pathlib import Path

ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "content" / "books" / "retrieval" / "_index.md").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError("repository root not found")

NOTEBOOK_DIR = ROOT / "notebooks" / "books" / "retrieval"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import _support
ROOT = _support.repo_root()
print("repo root:", ROOT)


In [ ]:
from document_identity import Collection
from evidence_units import build_units
from evidence_pipeline import identity, method_candidates, candidate_union, fuse, document_metrics, rerank, select

collection = Collection.load(ROOT / "examples" / "fixtures" / "part2-v1.json")
units = build_units(collection, "part2", "sections")
print("units:", len(units))
assert len(units) > 0

In [ ]:
a = next(u for u in units if u["source_id"] == "toy:blog")
b = next(u for u in units if u["source_id"] == "toy:spec")
c = next(u for u in units if u["document_id"] == "b-focused")
target = next(u for u in units if u["document_id"] == "scoped-reset")

bm25_hits = [{**u, "score": 3 - i} for i, u in enumerate([a, c])]
dense_hits = [{**u, "score": 3 - i} for i, u in enumerate([c, b])]
lists = {
    "bm25": method_candidates(bm25_hits, "bm25", "part2", "hand-test"),
    "dense": method_candidates(dense_hits, "dense", "part2", "hand-test"),
}
pool = candidate_union(lists)
print("pool size:", len(pool))
print("origins for shared candidate:", len(next(x for x in pool if identity(x["unit"]) == identity(c))["origins"]))
assert len(pool) == 3
assert len(next(x for x in pool if identity(x["unit"]) == identity(c))["origins"]) == 2

In [ ]:
# Candidate recall is an upper bound on later selection.
positives = [c["document_id"], "absent-doc"]
pool_metrics = document_metrics(positives, pool)
fused = fuse(lists)
fused_metrics = document_metrics(positives, fused)
print("pool candidate_recall:", pool_metrics["candidate_recall"])
print("fused recall@3:", fused_metrics["recall@3"])
assert pool_metrics["candidate_recall"] == 0.5
assert fused_metrics["recall@3"] <= pool_metrics["candidate_recall"]

In [ ]:
# A missing pool member cannot be recovered by later ordering or selection.
pool_ids = {x["unit"]["passage_id"] for x in pool}
assert target["passage_id"] not in pool_ids

score_envelopes = {
    x["unit"]["passage_id"]: {
        "score": float(i),
        "model_fingerprint": "manual-pair-control",
        "query_sha256": __import__("hashlib").sha256(b"query").hexdigest(),
        "text_sha256": __import__("hashlib").sha256(x["unit"]["text"].encode()).hexdigest(),
    }
    for i, x in enumerate(pool)
}
ranked = rerank(pool, score_envelopes, "manual-pair-control", "query")
selected = select(ranked, max_units=3)
selected_ids = {x["unit"]["passage_id"] for x in selected["selected"]}
assert target["passage_id"] not in {x["unit"]["passage_id"] for x in ranked}
assert selected_ids <= pool_ids
print("selected ids subset of pool:", selected_ids <= pool_ids)

## Observed versus not established
Observed: union, provenance, ceiling and non-recovery on constructed cases. Not established: general failure rates, remedies or downstream helpfulness.